In [ ]:
# streamflow_finetune.py
# Fine-tune pretrained encoder on labeled stream flow data

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import pandas as pd

# === Config ===
data_dir = 'data/labeled_stream_images'
labels_csv = 'data/labels.csv'  # CSV format: filename,label
pretrained_path = 'simclr_encoder.pth'
num_classes = 5
batch_size = 32
epochs = 30
learning_rate = 1e-4

# === Dataset ===
class StreamFlowDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.data.iloc[idx, 0])
        image = Image.open(img_path).convert('RGB')
        label = int(self.data.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

# === Transforms ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# === DataLoader ===
dataset = StreamFlowDataset(labels_csv, data_dir, transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# === Model Setup ===
encoder = models.resnet18()
encoder.fc = nn.Identity()
encoder.load_state_dict(torch.load(pretrained_path))

class StreamClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

model = StreamClassifier(encoder, num_classes)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# === Training ===
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100. * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f} Accuracy: {acc:.2f}%")

# === Save final model ===
torch.save(model.state_dict(), 'streamflow_classifier.pth')
print("Saved fine-tuned model to streamflow_classifier.pth")
